# VulnFix evaluation harness (GPU)

Runs the full pipeline over all 11 cases, N times each, and scores every patch
against independent oracles: does it compile, is it still exploitable, does it
still behave, and what did the validator say.

The headline number is the gap between **validated** and **truly fixed**.

Setup is the same as the demo notebook. Copy & Edit, GPU T4 x2, Internet on, Run All.

## 1. Code + tools

In [ ]:
%cd /kaggle/working
!rm -rf vulnfix && git clone -q https://github.com/BinuShefieldShifani/vulnfix.git
%cd vulnfix
!pip install -q -r requirements-dev.txt > /tmp/pip.log 2>&1 && echo "deps ok" || tail -20 /tmp/pip.log
!apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y zstd pciutils lshw default-jdk-headless >/dev/null 2>&1 && echo "system ok"

## 2. Ollama + models

In [ ]:
import subprocess, time, os
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama.log 2>&1 && echo "ollama installed"
subprocess.Popen("nohup ollama serve > /kaggle/working/ollama.log 2>&1 &", shell=True)
time.sleep(8)
for m in ["nomic-embed-text","qwen2.5-coder:7b","llama3.1:8b"]:
    r = subprocess.run(["ollama","pull",m], capture_output=True, text=True)
    print(m, "ready" if r.returncode==0 else "FAILED")

## 3. ChromaDB + advisories

In [ ]:
os.environ["ANONYMIZED_TELEMETRY"]="False"
subprocess.Popen("nohup chroma run --path /kaggle/working/chroma_data --host localhost --port 8000 > /kaggle/working/chroma.log 2>&1 &", shell=True)
time.sleep(8)
!python ingest_osv.py --ecosystems PyPI Maven NuGet --limit 500 | tail -1

## 4. Run the evaluation

5 repeats per case. This is the long cell: roughly 40 to 70 seconds per run,
11 cases, 5 repeats, so plan for 40 to 70 minutes. Progress prints per run.
Lower `repeats` for a quick smoke test.

In [ ]:
from eval.harness import run_evaluation
summary = run_evaluation(repeats=5, run_label="run5")

## 5. Show the summary

In [ ]:
import pandas as pd
df = pd.DataFrame(summary)
pd.set_option("display.max_columns", None)
display(df)

# the headline: where the validator disagrees with reality
gap = df[(df["validated_but_exploitable"].fillna(0) > 0) |
         (df["validated_but_broken"].fillna(0) > 0)]
print("\nCases where 'validated' hid a real problem:")
display(gap[["case_id","validated_rate","truly_fixed_rate","behavior_ok_rate",
             "validated_but_exploitable","validated_but_broken"]])

## 6. Package results to download

In [ ]:
!cd /kaggle/working/vulnfix && zip -rq /kaggle/working/eval_run5.zip results/eval/run5
!ls -lh /kaggle/working/eval_run5.zip
print("Download eval_run5.zip from the Output panel, then we add it to the repo.")